In [1]:
from tuya_connector import TuyaOpenAPI

ACCESS_ID = "8dued99qjs7sryafpa8c"
ACCESS_KEY = "173154a0470d4661802c6aafb94a178f"
API_ENDPOINT = "https://openapi.tuyaeu.com"
DEVICE_ID = "bf0178074ad9c548aduc79"

openapi = TuyaOpenAPI(API_ENDPOINT, ACCESS_ID, ACCESS_KEY)
openapi.connect()

# Get all status data
response = openapi.get(f"/v1.0/devices/{DEVICE_ID}/status")
print(response)

{'result': [{'code': 'total_forward_energy', 'value': 97736}, {'code': 'phase_a', 'value': 'CNkABkAAAQE='}, {'code': 'fault', 'value': 0}, {'code': 'switch_prepayment', 'value': False}, {'code': 'energy_reset', 'value': ''}, {'code': 'balance_energy', 'value': 0}, {'code': 'charge_energy', 'value': 0}, {'code': 'leakage_current', 'value': 0}, {'code': 'switch', 'value': True}, {'code': 'alarm_set_1', 'value': 'AwEADQQBAB4='}, {'code': 'alarm_set_2', 'value': 'AQEAPwMBAQ4EAQCq'}, {'code': 'breaker_number', 'value': '\x0e'}, {'code': 'leakagecurr_test', 'value': False}], 'success': True, 't': 1775246504706, 'tid': 'f07301392f9711f1b270d6c5c1a9e32b'}


In [2]:
import base64
import time
import csv
from tuya_connector import TuyaOpenAPI

# Tuya IoT credentials
ACCESS_ID = "8dued99qjs7sryafpa8c"
ACCESS_KEY = "173154a0470d4661802c6aafb94a178f"
API_ENDPOINT = "https://openapi.tuyaeu.com"
DEVICE_ID = "bf0178074ad9c548aduc79"

# Initialize API
openapi = TuyaOpenAPI(API_ENDPOINT, ACCESS_ID, ACCESS_KEY)
openapi.connect()

CSV_FILE = "project_data.csv"

# Create CSV file with header
with open(CSV_FILE, mode='w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow([
        "Time",
        "Voltage(V)",
        "Current(A)",
        "Power(W)",
        "Leakage Current(mA)",
        "Total Energy(kWh)",
        "Price(DZD)"
    ])

# Electricity tariff (price per kWh in Algeria)
TARIF_DZD = 5.34  

def get_measurements():
    voltage = current = power = energy = leakage = None
    response = openapi.get(f"/v1.0/devices/{DEVICE_ID}/status")

    for item in response.get("result", []):
        code = item["code"]

        # Phase A decoding for V/I/P
        if code == "phase_a":
            raw_bytes = base64.b64decode(item["value"])
            voltage_raw = raw_bytes[2] | (raw_bytes[3] << 8)
            current_raw = raw_bytes[4] | (raw_bytes[5] << 8)
            power_raw   = raw_bytes[6] | (raw_bytes[7] << 8)

            voltage = voltage_raw / 10.0
            current = current_raw / 100.0
            power   = power_raw / 10.0

        # Total energy (kWh)
        elif code == "total_forward_energy":
            energy = item["value"] / 100.0  

        # Leakage current
        elif code == "leakage_current":
            leakage = item["value"]

    return voltage, current, power, leakage, energy

# Collect and save data every 10 seconds
while True:
    voltage, current, power, leakage, energy = get_measurements()
    now = time.strftime("%Y-%m-%d %H:%M:%S")

    if voltage is not None and energy is not None:
        price = energy * TARIF_DZD  # 💡 Price = Energy × Tariff

        print(f"[{now}] V={voltage:.1f}V | I={current:.2f}A | "
            f"P={power:.1f}W | L={leakage}mA | "
            f"E={energy:.2f}kWh | Price={price:.2f} DZD")

        with open(CSV_FILE, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([now, voltage, current, power, leakage, energy, price])
    else:
        print(f"[{now}] No data found.")

    time.sleep(10)

[2026-04-03 22:02:03] V=153.6V | I=0.74A | P=6528.0W | L=0mA | E=977.36kWh | Price=5219.10 DZD
[2026-04-03 22:02:13] V=153.6V | I=2.54A | P=1459.3W | L=0mA | E=977.37kWh | Price=5219.16 DZD
[2026-04-03 22:02:23] V=153.6V | I=2.54A | P=1484.9W | L=0mA | E=977.37kWh | Price=5219.16 DZD
[2026-04-03 22:02:33] V=153.6V | I=2.54A | P=1484.9W | L=0mA | E=977.37kWh | Price=5219.16 DZD
[2026-04-03 22:02:43] V=179.2V | I=0.08A | P=1484.9W | L=0mA | E=977.37kWh | Price=5219.16 DZD
[2026-04-03 22:02:54] V=153.6V | I=2.54A | P=1408.1W | L=0mA | E=977.37kWh | Price=5219.16 DZD
[2026-04-03 22:03:04] V=153.6V | I=2.44A | P=1408.1W | L=0mA | E=977.37kWh | Price=5219.16 DZD
[2026-04-03 22:03:14] V=153.6V | I=2.44A | P=1356.9W | L=0mA | E=977.37kWh | Price=5219.16 DZD
[2026-04-03 22:03:24] V=153.6V | I=2.44A | P=1356.9W | L=0mA | E=977.37kWh | Price=5219.16 DZD
[2026-04-03 22:03:34] V=153.6V | I=2.44A | P=1356.9W | L=0mA | E=977.37kWh | Price=5219.16 DZD
[2026-04-03 22:03:44] V=153.6V | I=2.34A | P=1356.

KeyboardInterrupt: 